# 주가예측 캡스톤 — 학습 + 백테스트 (Kaggle)

**셀을 위에서부터 순서대로 실행하면 끝난다.** 중간에 고칠 것 없다.

매번 코드를 GitHub에서 새로 받으므로 항상 최신이다.
학습과 백테스트를 같은 세션에서 돌리므로 체크포인트를 주고받을 필요가 없다.

---
### 실행 전 우측 패널에서 3가지 설정

| 설정 | 값 |
|---|---|
| **Accelerator** | `GPU T4 x2` |
| **Internet** | `On` |
| **Input** → Add Input | 업로드해둔 데이터셋 |

> 이 3가지가 회색으로 안 눌리면 **휴대폰 인증**이 필요하다.
> Settings → Phone Verification (한 번만 하면 된다)

예상 소요: **약 15분** (학습 10분 + 백테스트 2분)


## 1. GPU 확인
`cuda: True` 가 나와야 한다. False 면 Accelerator 를 GPU 로 바꾸고 세션 재시작.


In [ ]:
import torch

ok = torch.cuda.is_available()
print('cuda:', ok)
print('GPU :', torch.cuda.get_device_name(0) if ok else '없음')
if not ok:
    print()
    print('⚠️ 우측 Accelerator 를 GPU T4 x2 로 바꾸고 Run → Restart Session')


## 2. 코드 + 데이터 준비

코드를 항상 새로 받는다(기존 폴더는 지운다). 데이터는 Input 에서 자동으로 찾는다.


In [ ]:
import glob, os, shutil, zipfile, subprocess

# --- 코드: 항상 새로 받는다 (예전 버전이 남아 헷갈리는 일을 막는다)
os.chdir('/kaggle/working')
shutil.rmtree('Capstone_Stock_Price_Prediction', ignore_errors=True)
subprocess.run(['git', 'clone', '-q',
                'https://github.com/sungjunpk/Capstone_Stock_Price_Prediction.git'],
               check=True)
os.chdir('/kaggle/working/Capstone_Stock_Price_Prediction')
print('최신 코드:', subprocess.run(['git','log','-1','--format=%h %s'],
                                capture_output=True, text=True).stdout.strip())

# --- 데이터: Input 에서 zip 이든 parquet 이든 찾아서 푼다
os.makedirs('data/processed', exist_ok=True)
zips  = glob.glob('/kaggle/input/**/*.zip', recursive=True)
parqs = glob.glob('/kaggle/input/**/panel.parquet', recursive=True)

if zips:
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall('.')
    print('데이터:', zips[0])
elif parqs:
    src = os.path.dirname(parqs[0])
    for f in glob.glob(f'{src}/*.parquet'):
        shutil.copy(f, 'data/processed/')
    print('데이터:', src)
else:
    raise SystemExit('❌ 데이터를 못 찾았다 — 우측 Add Input 을 확인할 것')

print('준비된 파일:', sorted(os.listdir('data/processed')))

# --- 설정 확인: 낡은 코드로 도는 사고를 여기서 잡는다
import yaml
enc = yaml.safe_load(open('configs/config.yaml'))['model']['encoder']
print(f"\n모델 설정: d_model={enc['d_model']} n_layers={enc['n_layers']} dropout={enc['dropout']}")
assert enc['d_model'] == 32 and enc['n_layers'] == 1, (
    "❌ 옛날 설정이다. 학습 로그에 '파라미터 1.88M' 이 뜨면 이 노트북이 아니라 "
    "예전 노트북을 돌리고 있는 것 — 이 셀부터 다시 실행할 것.")
print("✅ 확정 설정(minimal, 0.33M) 맞음")


## 3. 배관 점검 (1분)

전체 학습 전에 소규모로 한 번 돌려 본다. 여기서 에러가 나면 아래도 다 실패한다.


In [ ]:
!python scripts/train.py --smoke


## 4. 학습 (약 10분)

설정은 `configs/config.yaml` 에 이미 최적값이 들어 있다(스윕으로 찾은 minimal).

**볼 지점**: 마지막에 `✅ 기준선 대비 N% 개선` 이 나와야 한다.
`❌ 기준선을 못 이겼다` 면 모델이 아무것도 학습하지 못한 것이다.


In [ ]:
!python scripts/train.py --batch-size 512


## 5. 백테스트 + 규칙 비교 (3분)

예측을 한 번만 계산하고 매매 규칙만 갈아끼워 5가지를 비교한다. 학습은 다시 안 한다.

| 변형 | 답하는 질문 |
|---|---|
| 버퍼없음 | 기준점 (이전 실행과 같은 규칙) |
| +버퍼 | 밀려난 종목을 바로 안 팔면 회전율이 얼마나 주나 |
| +버퍼+밴드 | 잔챙이 거래까지 막으면 |
| +익절해제 | 익절 10%가 상승장에서 승자를 자르고 있었나 |
| absolute | 순위 전환의 효과 (대조군) |

**성과보다 `예측력 진단` 을 먼저 읽을 것.** 랭크 IC 의 `t` 가 2 미만이면
매매 규칙을 어떻게 바꿔도 성과는 안 나온다.

> 이 셀은 앞으로 안 바뀐다. 규칙을 바꿔도 셀 2가 저장소를 새로 클론하므로
> **캐글에서는 재시작만 하면 된다.**

In [ ]:
!python scripts/backtest.py --compare

## 6. 학습 요약

백테스트 비교표는 위 셀 출력에 있다. 여기는 학습 쪽만 다시 모아 본다.

In [ ]:
import glob, json

tr = sorted(g for g in glob.glob('outputs/reports/*.json')
            if 'backtest' not in g and 'sweep' not in g and 'smoke' not in g)
if tr:
    r = json.load(open(tr[-1]))
    print(f"기준선   {r['baseline_val_loss']:.6f}")
    print(f"best val {r['best_val_loss']:.6f}  (epoch {r['best_epoch']})")
    print(f"개선     {r['improvement_vs_baseline_pct']:+.2f}%")
    print()
    print('피처 중요도 상위 8 — 해석가능성 리포트의 근거')
    for k, v in list(r['feature_importance'].items())[:8]:
        print(f'  {k:16s} {v:.4f}')
else:
    print('학습 리포트를 못 찾았다 — 4번 셀을 먼저 실행할 것')

## 7. 결과 내려받기

실행하면 우측 **Output** 패널에 `phase1_result.zip` 이 생긴다.
받아서 로컬 저장소의 `outputs/` 에 풀면 된다.


In [ ]:
import os, shutil

shutil.rmtree('/kaggle/working/download', ignore_errors=True)
os.makedirs('/kaggle/working/download', exist_ok=True)
for p in ['outputs/checkpoints', 'outputs/reports']:
    if os.path.exists(p):
        shutil.copytree(p, f'/kaggle/working/download/{os.path.basename(p)}')

shutil.make_archive('/kaggle/working/phase1_result', 'zip', '/kaggle/working/download')
print('→ 우측 Output 패널에서 phase1_result.zip 다운로드')
!ls -lh /kaggle/working/phase1_result.zip


---
## (선택) 모델 크기 더 탐색하기

지금 설정으로 결과가 아쉬우면 여러 크기를 비교해 본다. 약 20분.
작을수록 좋았으므로 더 작은 쪽(nano, pico)을 확인하는 것이다.


In [ ]:
!python scripts/sweep.py --epochs 15 --batch-size 512 --only minimal,nano,pico


---
## 문제 해결

| 증상 | 해결 |
|---|---|
| `cuda: False` | Accelerator 를 GPU 로 → Run → Restart Session |
| `데이터를 못 찾았다` | 우측 **Add Input** 으로 데이터셋을 붙인다 |
| `git clone` 이 멈춤 | **Internet: On** 확인 |
| 설정이 회색으로 안 눌림 | 휴대폰 인증 (Settings → Phone Verification) |
| `CUDA out of memory` | `--batch-size` 를 512 → 256 → 128 |
| 세션이 끊김 | 무료 GPU 는 세션 12시간 / 주당 30시간 |

**데이터를 새로 만들었을 때**는 로컬에서 `python scripts/package_data.py` 를 돌리고
Kaggle 데이터셋 페이지에서 **New Version** 으로 새 zip 을 올린다.

**수집은 로컬에서만 한다** — 키움 API 는 등록된 IP 에서만 호출된다.
여기에는 API 키(`.env`)가 필요 없다.
